# 04 — Validation : Nelson-Siegel vs Svensson sur la courbe des taux BCE

Question : 4 ou 6 paramètres suffisent-ils à reproduire les taux spot AAA de la BCE, par maturité et par régime ?

Conventions : résidu = observé − modèle · erreurs en points de base (pb) · tranches de maturité ]lo, hi] avec bornes [0, 2, 5, 10, 20, 30] · régimes *avant* / *après* la première hausse BCE du 27/07/2022 · seuil d'erreur = max(0,1 % / T, plancher de 4 pb).

Les cellules d'ajustement longues (≈ 2 min 30) sont commentées.

## 1. Données et configuration

In [2]:
import sys
sys.path.append("..")          # ajoute la racine du projet

from src.validation.splits import choisir_caches
from src.validation.residue import rmse

from src.curves.data import charger_courbe

from src.curves.nelson_siegel import fit_ns, ns_rate
from src.curves.svensson import fit_sv, sv_rate
from src.curves.evaluation import evaluer_date, ajuster_historique


import pandas as pd

In [3]:
chemin_yield="../data/raw/yield_curve.csv"
chemin_param="../data/raw/param.csv"
courbe=charger_courbe(chemin_yield)

bornes = [0, 2, 5, 10, 20, 30]
maturites = pd.Series(courbe.columns, index=courbe.columns)   # l'index EST la maturité
caches, ancrage = choisir_caches(maturites, bornes)
masque_cache = courbe.columns.isin(caches)

## 2. Maturités cachées (test sur le 28/10/2026)
Controle de la liste figée de maturités cachées, puis comparaison NS ajusté sur l'ancrage / sur toute la courbe.

In [4]:
#test pour valider que ca fonctionne bien
caches, ancrage = choisir_caches(maturites, bornes)

assert len(caches) == 15                           # 5 tranches x 3 parts
assert len(caches) + len(ancrage) == courbe.shape[1]
assert courbe.columns.min() in ancrage             # 3 mois gardé en ancrage
assert courbe.columns.max() in ancrage             # 30 ans gardé en ancrage
print(sorted(round(c, 2) for c in caches))

[np.float64(0.33), np.float64(1.17), np.float64(2.0), np.float64(2.67), np.float64(3.42), np.float64(5.0), np.float64(5.33), np.float64(7.75), np.float64(8.67), np.float64(10.33), np.float64(15.0), np.float64(20.0), np.float64(22.42), np.float64(25.75), np.float64(29.17)]


In [5]:
T = courbe.columns.values
date = "2026-10-28"
y = courbe.loc[date].values

ev_ns = evaluer_date(T, y, masque_cache, fit_ns, ns_rate)
ev_sv = evaluer_date(T, y, masque_cache, fit_sv, sv_rate)

pd.DataFrame({"Nelson-Siegel": [ev_ns["rmse_ancrage"], ev_ns["rmse_cache"]],
              "Svensson":      [ev_sv["rmse_ancrage"], ev_sv["rmse_cache"]]},
             index=["ancrage", "cachés"]) * 1e4      # en points de base

,Nelson-Siegel,Svensson
ancrage,3.312118,0.000028
cachés,7.023368,0.000032


In [6]:
# NS ajusté sur TOUTE la courbe, puis RMSE aux seules maturités cachées
res_tout = fit_ns(T, y)
rmse_tout = rmse((y - ns_rate(T, *res_tout.x))[masque_cache])

print("cachés, ajusté sur l'ancrage (pb)  :", round(ev_ns["rmse_cache"] * 1e4, 2))
print("mêmes maturités, ajusté sur tout (pb):", round(rmse_tout * 1e4, 2))

cachés, ajusté sur l'ancrage (pb)  : 7.02
mêmes maturités, ajusté sur tout (pb): 6.98


## 3. Ajustement date par date
### 3.1 Essai rapide sur 20 dates

In [7]:

NOMS_NS = ["beta0", "beta1", "beta2", "tau"]
NOMS_SV = ["beta0", "beta1", "beta2", "beta3", "tau1", "tau2"]

# essai rapide
essai = ajuster_historique(courbe.iloc[:20], fit_sv, sv_rate, NOMS_SV)
print(essai[["rmse_pb", "succes", "repli"]].describe())


         rmse_pb
count  20.000000
mean    0.000036
std     0.000017
min     0.000027
25%     0.000028
50%     0.000029
75%     0.000031
max     0.000090


### 3.2 Run complet NS et Svensson (≈ 2 min 30)
Bloc commenté : le décommenter pour régénérer `hist_ns.csv` et `hist_sv.csv`.

In [8]:

# NOMS_NS = ["beta0", "beta1", "beta2", "tau"]
# NOMS_SV = ["beta0", "beta1", "beta2", "beta3", "tau1", "tau2"]

# # run complet
# hist_ns = ajuster_historique(courbe, fit_ns, ns_rate, NOMS_NS)
# hist_ns.to_csv("../data/processed/hist_ns.csv")

# hist_sv = ajuster_historique(courbe, fit_sv, sv_rate, NOMS_SV)
# hist_sv.to_csv("../data/processed/hist_sv.csv")

## 4. Synthèse de l'ajustement
RMSE par date de chaque modèle (toute la courbe), nombre de replis, dates mal ajustées.

In [9]:
#synthèse des résultats
import pandas as pd
from src.curves.synthese import synthese_historique, rmse_par_annee

ns = pd.read_csv("../data/processed/hist_ns.csv", index_col=0, parse_dates=True)
sv = pd.read_csv("../data/processed/hist_sv.csv", index_col=0, parse_dates=True)

tableau = pd.concat({"NS": synthese_historique(ns, ["tau"]),
                     "SV": synthese_historique(sv, ["tau1", "tau2"])},
                    axis=1, sort=False)      # sort=False : garde l'ordre des lignes
print(tableau.to_string())
print(rmse_par_annee(ns, sv))

                                            NS          SV
dates                                     5662        5662
RMSE moyenne (pb)                     3.423589    0.000321
RMSE médiane (pb)                     2.274705    0.000029
RMSE quantile 95 % (pb)              10.754095     0.00015
RMSE max (pb)                        16.116596    0.048178
date du max                         2012-03-29  2018-02-02
dates RMSE > 0.05 pb                      5388           0
dates RMSE > 5 pb                         1315           0
dates RMSE > 10 pb                         455           0
succes = False                               0          54
replis                                       0          34
dates |beta| > 20 % (non lisible)           55        1359
tau > 20 ans                                83         NaN
tau1 > 20 ans                              NaN           0
tau2 > 20 ans                              NaN           0
tau1/tau2 entre 0,77 et 1,3                NaN        25

## 5. Résidus (observé − modèle)
Résidus recalculés à partir des paramètres de `hist_ns.csv` et `hist_sv.csv`, avec contrôle de la RMSE.

In [10]:
from src.curves.evaluation import residus_historique
from src.curves.nelson_siegel import ns_rate
from src.curves.svensson import sv_rate

NOMS_NS = ["beta0", "beta1", "beta2", "tau"]
NOMS_SV = ["beta0", "beta1", "beta2", "beta3", "tau1", "tau2"]

ns = pd.read_csv("../data/processed/hist_ns.csv", index_col=0, parse_dates=True)
sv = pd.read_csv("../data/processed/hist_sv.csv", index_col=0, parse_dates=True)

res_ns = residus_historique(courbe, ns, ns_rate, NOMS_NS)
res_sv = residus_historique(courbe, sv, sv_rate, NOMS_SV)

# contrôle : on doit retrouver les RMSE de hist_*.csv (en pb)
import numpy as np
print(np.allclose(np.sqrt((res_ns ** 2).mean(axis=1)) * 1e4, ns["rmse_pb"]))
print(np.allclose(np.sqrt((res_sv ** 2).mean(axis=1)) * 1e4, sv["rmse_pb"]))

# contrôle : on lit bien le fichier à grille large (attendu : max ≈ 0,048 pb et 0 date > 0,05 pb)
print(sv["rmse_pb"].max(), (sv["rmse_pb"] > 0.05).sum())


True
True
0.0481776284452059 0


## 6. Biais, écart-type et RMSE par tranche et par régime

In [11]:
#calcul du biais et de la variance par tranche
BORNES = [0, 2, 5, 10, 20, 30]
DATE_HAUSSE = pd.Timestamp("2022-07-27")     #hausse des taux

from src.validation.residue import stats_par_tranche

def stats_par_regime(res):
    avant = res[res.index < DATE_HAUSSE]
    apres = res[res.index >= DATE_HAUSSE]
    assert len(avant) + len(apres) == len(res)
    return pd.concat({"avant": stats_par_tranche(avant, BORNES),
                      "après": stats_par_tranche(apres, BORNES)}, axis=1) * 1e4   # en pb

tab_ns = stats_par_regime(res_ns)
print(tab_ns.round(2))

# contrôle de l'identité rmse² = biais² + écart-type²
for reg in ["avant", "après"]:
    t = tab_ns[reg]
    assert np.allclose(t["rmse"]**2, t["biais"]**2 + t["ecart_type"]**2, rtol=1e-9, atol=0)

         avant                  après                 
         biais ecart_type  rmse biais ecart_type  rmse
]0, 2]    0.63       5.76  5.79  1.15       7.63  7.72
]2, 5]   -0.55       4.31  4.34 -0.66       3.75  3.81
]5, 10]  -1.97       3.58  4.08 -2.92       1.83  3.44
]10, 20]  2.36       4.39  4.98  2.86       2.40  3.73
]20, 30] -1.46       4.96  5.17 -1.61       4.26  4.56


## 7. Seuil de tolérance et dépassements
Seuil(T) = max(0,1 % / T, plancher) ; ratio = |résidu| / seuil ; dépassement = ratio > 1.

In [12]:
from src.curves.evaluation import ratio_erreur_seuil

TOLERANCE_PRIX = 0.001     # 0,1 % du prix d'un flux (fixée avant les résultats)
PLANCHER_PB = 4            # ancrage BCE, ajouté après les premiers résultats NS

ratio_ns = ratio_erreur_seuil(res_ns, TOLERANCE_PRIX, PLANCHER_PB)
ratio_sv = ratio_erreur_seuil(res_sv, TOLERANCE_PRIX, PLANCHER_PB)


COLS_PB = ["biais", "ecart_type", "rmse"]          # seules colonnes en pb

def stats_par_regime(res, ratio=None):
    if ratio is not None and not ratio.index.equals(res.index):
        raise ValueError("ratio et res doivent avoir les mêmes dates.")
    avant = np.asarray(res.index < DATE_HAUSSE)    # un seul masque pour les deux matrices
    blocs = {}
    for nom, m in [("avant", avant), ("après", ~avant)]:
        t = stats_par_tranche(res[m], BORNES, None if ratio is None else ratio[m])
        t[COLS_PB] = t[COLS_PB] * 1e4              # pb ; "depassement" reste entre 0 et 1
        blocs[nom] = t
    return pd.concat(blocs, axis=1)

tab_ns = stats_par_regime(res_ns, ratio_ns)
tab_sv = stats_par_regime(res_sv, ratio_sv)
print(tab_ns.round(3))
print(tab_sv.round(4))

          avant                                après                    \
          biais ecart_type   rmse depassement  biais ecart_type   rmse   
]0, 2]    0.627      5.757  5.791       0.094  1.153      7.631  7.718   
]2, 5]   -0.547      4.307  4.342       0.228 -0.665      3.753  3.811   
]5, 10]  -1.965      3.576  4.081       0.183 -2.915      1.827  3.441   
]10, 20]  2.360      4.390  4.985       0.206  2.857      2.403  3.733   
]20, 30] -1.465      4.962  5.173       0.175 -1.609      4.264  4.558   

                      
         depassement  
]0, 2]         0.122  
]2, 5]         0.249  
]5, 10]        0.295  
]10, 20]       0.346  
]20, 30]       0.333  
           avant                                  après                     \
           biais ecart_type    rmse depassement   biais ecart_type    rmse   
]0, 2]   -0.0000     0.0039  0.0039         0.0  0.0001     0.0091  0.0091   
]2, 5]    0.0001     0.0027  0.0027         0.0 -0.0002     0.0068  0.0068   
]5, 10] 

### 7.1 Sensibilité au plancher (0, 3, 4, 5 pb ; 4 pb retenu)

In [13]:
def sensibilite_plancher(res, planchers, tolerance_prix=TOLERANCE_PRIX):
    """Proportion de dépassements par tranche et par régime, pour plusieurs planchers (pb).

    res : résidus (dates x maturités). planchers : liste de valeurs en pb ; 0 = sans plancher.
    Retourne un DataFrame : une ligne par tranche, colonnes (plancher_pb, régime).
    """
    resultats = {}
    for p in planchers:
        ratio = ratio_erreur_seuil(res, tolerance_prix, p)          # recalculé à chaque plancher
        tab = stats_par_regime(res, ratio)
        resultats[p] = tab.xs("depassement", axis=1, level=1)       # garde "depassement" par nom
    return pd.concat(resultats, axis=1, names=["plancher_pb", "régime"])

PLANCHERS = [0, 3, 4, 5]        # fixés avant de regarder les résultats ; 4 = valeur retenue

sens_ns = sensibilite_plancher(res_ns, PLANCHERS)
sens_sv = sensibilite_plancher(res_sv, PLANCHERS)
print(sens_ns.round(3))
print(sens_sv.round(4))

plancher_pb      0             3             4             5       
régime       avant  après  avant  après  avant  après  avant  après
]0, 2]       0.094  0.122  0.094  0.122  0.094  0.122  0.094  0.122
]2, 5]       0.299  0.401  0.269  0.347  0.228  0.249  0.184  0.153
]5, 10]      0.440  0.843  0.218  0.500  0.183  0.295  0.152  0.120
]10, 20]     0.462  0.814  0.223  0.490  0.206  0.346  0.189  0.205
]20, 30]     0.523  0.894  0.203  0.462  0.175  0.333  0.152  0.243
plancher_pb     0           3           4           5      
régime      avant après avant après avant après avant après
]0, 2]        0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0
]2, 5]        0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0
]5, 10]       0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0
]10, 20]      0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0
]20, 30]      0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0


## 8. Répartition des erreurs (quantiles de |résidu|)

In [14]:
#on regarde la répartition des erreurs
from src.validation.residue import quantiles_par_tranche

def quantiles_par_regime(res):
    avant = np.asarray(res.index < DATE_HAUSSE)
    return pd.concat({"avant": quantiles_par_tranche(res[avant], BORNES) * 1e4,
                      "après": quantiles_par_tranche(res[~avant], BORNES) * 1e4}, axis=1)   # en pb

q_ns = quantiles_par_regime(res_ns)
print(q_ns.round(2))

q_sv = quantiles_par_regime(res_sv)
print(q_sv.round(4))


         avant                     après                    
           q50   q75    q90    q95   q50   q75    q90    q95
]0, 2]    2.34  5.92  10.02  11.97  3.46  6.53  11.88  16.13
]2, 5]    1.28  3.59   7.74  10.41  2.33  4.07   5.79   7.48
]5, 10]   1.12  2.50   7.59  10.95  3.00  4.21   5.17   5.67
]10, 20]  0.63  2.16  10.47  13.03  2.93  4.66   6.09   6.81
]20, 30]  0.44  1.80   8.35  13.47  2.74  4.91   7.86   9.65
         avant                      après                  
           q50  q75     q90     q95   q50  q75  q90     q95
]0, 2]     0.0  0.0  0.0001  0.0002   0.0  0.0  0.0  0.0001
]2, 5]     0.0  0.0  0.0001  0.0002   0.0  0.0  0.0  0.0001
]5, 10]    0.0  0.0  0.0001  0.0001   0.0  0.0  0.0  0.0001
]10, 20]   0.0  0.0  0.0001  0.0001   0.0  0.0  0.0  0.0001
]20, 30]   0.0  0.0  0.0001  0.0001   0.0  0.0  0.0  0.0001


## 9. Autocorrélation des résidus
ρ à un retard et n_eff = n(1 − ρ)/(1 + ρ) par tranche et par régime.

In [15]:
# test des écarts avec l'autocorrélation 

from src.validation.residue import autocorrelation_par_tranche

def autocorr_par_regime(res):
    avant = np.asarray(res.index < DATE_HAUSSE)
    return pd.concat({"avant": autocorrelation_par_tranche(res[avant], BORNES),
                      "après": autocorrelation_par_tranche(res[~avant], BORNES)}, axis=1)

print(autocorr_par_regime(res_ns).round(3))

           avant                  après               
               n    rho   n_eff       n    rho   n_eff
]0, 2]    4574.0  0.974  60.740  1088.0  0.903  55.637
]2, 5]    4574.0  0.981  43.110  1088.0  0.900  57.543
]5, 10]   4574.0  0.992  19.398  1088.0  0.877  71.356
]10, 20]  4574.0  0.995  12.122  1088.0  0.925  42.107
]20, 30]  4574.0  0.994  14.449  1088.0  0.926  42.080


## 10. Test de biais robuste à l'autocorrélation (Newey-West)

In [16]:
from src.validation.tests import test_biais_par_tranche

def test_biais_par_regime(res, n_lags=None):
    avant = np.asarray(res.index < DATE_HAUSSE)
    t = pd.concat({"avant": test_biais_par_tranche(res[avant], BORNES, n_lags),
                   "après": test_biais_par_tranche(res[~avant], BORNES, n_lags)}, axis=1)
    for reg in ["avant", "après"]:
        t[(reg, "biais")] *= 1e4          # pb
        t[(reg, "se")] *= 1e4
    return t

# retards fixés avant de regarder les résultats
for L in [None, 20, 60]:
    print("n_lags =", L)
    print(test_biais_par_regime(res_ns, L)[[("avant","biais"),("avant","se"),("avant","t"),
                                            ("après","biais"),("après","se"),("après","t")]].round(2))

n_lags = None
         avant              après             
         biais    se      t biais    se      t
]0, 2]    0.63  0.06  10.14  1.15  0.11  10.87
]2, 5]   -0.55  0.15  -3.69 -0.66  0.21  -3.22
]5, 10]  -1.97  0.13 -15.42 -2.92  0.08 -34.99
]10, 20]  2.36  0.18  12.83  2.86  0.12  24.62
]20, 30] -1.46  0.12 -11.85 -1.61  0.09 -18.26
n_lags = 20
         avant              après             
         biais    se      t biais    se      t
]0, 2]    0.63  0.09   7.15  1.15  0.17   6.94
]2, 5]   -0.55  0.21  -2.59 -0.66  0.33  -2.01
]5, 10]  -1.97  0.18 -10.74 -2.92  0.13 -22.85
]10, 20]  2.36  0.26   8.91  2.86  0.19  15.32
]20, 30] -1.46  0.18  -8.24 -1.61  0.14 -11.47
n_lags = 60
         avant             après             
         biais    se     t biais    se      t
]0, 2]    0.63  0.14  4.37  1.15  0.23   4.95
]2, 5]   -0.55  0.34 -1.59 -0.66  0.50  -1.32
]5, 10]  -1.97  0.30 -6.47 -2.92  0.19 -15.75
]10, 20]  2.36  0.44  5.34  2.86  0.29  10.00
]20, 30] -1.46  0.30 -4.96 -

### 10.1 Convergence du t selon le nombre de retards

In [17]:
#t continue de baisser et n'a tjrs pas converger, il y a une plus forte dépéndances que 60 jours entre les taux
def t_selon_retards(res, retards):
    """Statistique t du test de biais par tranche, pour plusieurs nombres de retards."""
    return pd.concat({L: test_biais_par_tranche(res, BORNES, L)["t"] for L in retards},
                     axis=1)

RETARDS = [20, 60, 120, 250, 500]        # fixés avant de regarder les résultats

avant = np.asarray(res_ns.index < DATE_HAUSSE)
print(t_selon_retards(res_ns[avant], RETARDS).round(2))                # n = 4 574
print(t_selon_retards(res_ns[~avant], [20, 60, 120, 250]).round(2))    # n = 1 088

            20    60    120   250   500
]0, 2]     7.15  4.37  3.23  2.41  1.90
]2, 5]    -2.59 -1.59 -1.18 -0.90 -0.72
]5, 10]  -10.74 -6.47 -4.73 -3.45 -2.67
]10, 20]   8.91  5.34  3.92  2.88  2.21
]20, 30]  -8.24 -4.96 -3.66 -2.71 -2.09
            20     60     120    250
]0, 2]     6.94   4.95   4.23   3.77
]2, 5]    -2.01  -1.32  -1.04  -0.88
]5, 10]  -22.85 -15.75 -13.21 -12.85
]10, 20]  15.32  10.00   7.86   6.61
]20, 30] -11.47  -7.49  -5.92  -4.99


## 11. Diagnostic : biais par année et par blocs
Le biais « avant » est dominé par 2010-2014 ; découpage en blocs exploratoire, choisi après avoir vu le tableau annuel.

In [18]:
def stats_par_annee(res):
    """Biais, écart-type, RMSE (pb) par année, régime et tranche.

    La frontière DATE_HAUSSE coupe 2022 en deux groupes (avant / après les hausses).
    n_dates donne le nombre de dates du groupe.
    """
    regime = np.where(res.index < DATE_HAUSSE, "avant", "après")
    blocs = {}
    for (annee, reg), bloc in res.groupby([res.index.year, regime]): #on déocupe en blocs de date
        t = stats_par_tranche(bloc, BORNES)
        t[COLS_PB] = t[COLS_PB] * 1e4
        t["n_dates"] = len(bloc)
        blocs[(annee, reg)] = t
    return pd.concat(blocs, names=["annee", "regime", "tranche"])

an_ns = stats_par_annee(res_ns)
print(an_ns["biais"].unstack("tranche").round(2))     # biais : une ligne par (année, régime), une colonne par tranche

tranche       ]0, 2]  ]2, 5]  ]5, 10]  ]10, 20]  ]20, 30]
annee regime                                             
2004  avant    -0.01   -0.15     0.22     -0.12      0.05
2005  avant     0.14    0.10    -0.38      0.22     -0.10
2006  avant    -0.35    3.21    -1.75     -0.26      0.38
2007  avant    -0.43    2.19    -1.07     -0.09      0.16
2008  avant     0.16    0.26    -0.53      0.27     -0.12
2009  avant    -1.49    1.87     0.02     -0.28      0.14
2010  avant     2.28   -6.04    -1.91      6.52     -4.69
2011  avant     2.43   -5.51    -1.17      4.45     -3.12
2012  avant     3.01   -5.91    -6.39     11.01     -7.33
2013  avant     2.14   -1.59    -8.66      9.79     -5.91
2014  avant     1.15    0.62    -6.74      6.25     -3.56
2015  avant     0.47    0.31    -1.84      1.36     -0.69
2016  avant     0.54    0.07    -1.23      0.79     -0.36
2017  avant     0.64   -0.47    -0.81      0.71     -0.36
2018  avant     0.06   -0.02    -0.14      0.12     -0.07
2019  avant   

### 11.1 Test par blocs (2004-2009, 2010-2014, 2015-07/2022)

In [19]:
BLOCS = {"2004-2009":    (pd.Timestamp("2004-01-01"), pd.Timestamp("2010-01-01")),
         "2010-2014":    (pd.Timestamp("2010-01-01"), pd.Timestamp("2015-01-01")),
         "2015-07/2022": (pd.Timestamp("2015-01-01"), DATE_HAUSSE)}    # [début, fin[ : fin exclue

RETARDS = [20, 60, 120, 250]

def tests_par_blocs(res, blocs, retards):
    """Statistique t (par tranche, par bloc et par nombre de retards) et rho (par bloc)."""
    masques = {nom: np.asarray((res.index >= d) & (res.index < f)) for nom, (d, f) in blocs.items()}
    # contrôle : aucun chevauchement, aucune date de "avant" oubliée
    assert sum(m.sum() for m in masques.values()) == (res.index < DATE_HAUSSE).sum()
    assert np.all(sum(m.astype(int) for m in masques.values()) <= 1)
    t = pd.concat({(nom, L): test_biais_par_tranche(res[m], BORNES, L)["t"]
                   for nom, m in masques.items() for L in retards}, axis=1)
    rho = pd.concat({nom: autocorrelation_par_tranche(res[m], BORNES)["rho"]
                     for nom, m in masques.items()}, axis=1)
    return t, rho

t_blocs, rho_blocs = tests_par_blocs(res_ns, BLOCS, RETARDS)
print(t_blocs.round(2))
print(rho_blocs.round(3))

         2004-2009                   2010-2014                     \
               20    60    120   250       20     60    120   250   
]0, 2]       -3.38 -2.38 -1.98 -1.78     15.33  10.18  8.53  7.90   
]2, 5]        5.66  3.92  3.30  3.06     -7.48  -4.67 -3.65 -3.04   
]5, 10]      -5.67 -3.75 -3.03 -2.61    -11.38  -6.98 -5.24 -3.98   
]10, 20]     -0.61 -0.44 -0.40 -0.42     15.21   9.77  8.13  7.89   
]20, 30]      2.03  1.46  1.29  1.32    -14.01  -9.04 -7.65 -7.95   

         2015-07/2022                    
                  20    60    120   250  
]0, 2]           8.84  5.81  4.47  3.42  
]2, 5]           3.14  2.12  1.75  1.43  
]5, 10]        -10.90 -7.37 -6.17 -5.06  
]10, 20]         9.23  6.09  5.07  4.26  
]20, 30]        -7.28 -4.99 -4.24 -3.69  
          2004-2009  2010-2014  2015-07/2022
]0, 2]        0.918      0.958         0.943
]2, 5]        0.943      0.981         0.887
]5, 10]       0.957      0.989         0.962
]10, 20]      0.954      0.984         0.9

## 12. Svensson : grille de départs élargie
La grille de départs passe de 15 à 56 points (τ1 et τ2 de 0,25 à 30 ans) après l'échec de 42 dates ; ancien résultat conservé dans `hist_sv_grille15.csv` ; cellule d'ajustement commentée (≈ 2 min 30).

In [21]:
# #résultats post changement de grille
# hist_sv = ajuster_historique(courbe, fit_sv, sv_rate, NOMS_SV)
# hist_sv.to_csv("../data/processed/hist_sv.csv")
# print(hist_sv["rmse_pb"].describe(), hist_sv["repli"].sum())

In [22]:
hist_sv = pd.read_csv("../data/processed/hist_sv.csv", index_col=0, parse_dates=True)

## 13. Diagnostics Svensson vs paramètres BCE
Ordre de TAU1 et TAU2, région identique à la BCE, écart entre notre courbe et la courbe recalculée avec les paramètres BCE.

In [23]:
def ordre_tau(params, tol=1e-6):
    """Classe chaque date selon l'ordre de TAU1 et TAU2 dans les paramètres BCE.

    params : DataFrame (index = dates) avec les colonnes "TAU1" et "TAU2".
    tol : écart (en années) en dessous duquel TAU1 et TAU2 sont considérés égaux.
    Retourne une Series indexée par date : "tau1<tau2", "tau1>tau2" ou "egal".
    """
    ecart = params["TAU1"] - params["TAU2"]
    return pd.Series(np.where(ecart.abs() <= tol, "egal",
                              np.where(ecart < 0, "tau1<tau2", "tau1>tau2")),
                     index=params.index, name="ordre_tau")

In [24]:
from src.curves.data import charger_parametres

params = charger_parametres("../data/raw/param.csv")
ordre = ordre_tau(params)

T = courbe.columns.to_numpy(dtype=float)
noms_bce = ["BETA0", "BETA1", "BETA2", "BETA3", "TAU1", "TAU2"]
noms_sv = ["beta0", "beta1", "beta2", "beta3", "tau1", "tau2"]

#on recalcule les taux avec les paramètres
courbe_bce = pd.DataFrame(sv_rate(T, *[params[c].to_numpy()[:, None] for c in noms_bce]),
                          index=params.index, columns=T)
courbe_sv = pd.DataFrame(sv_rate(T, *[hist_sv[c].to_numpy()[:, None] for c in noms_sv]),
                           index=hist_sv.index, columns=T)



In [25]:
#controle sur le nombre de dates avecR MSE >0.05, les régions avec les meme régions de taux que la bce etl'écart entre courbe fit et bce recalc
print((hist_sv["rmse_pb"] > 0.05).sum())            # avant : 42
print((~hist_sv["succes"]).sum())                    # avant : 53

ordre_sv = ordre_tau(hist_sv.rename(columns={"tau1": "TAU1", "tau2": "TAU2"}))
print((ordre_sv == ordre).mean())                  # avant : 0,882 (même région que la BCE)

courbe_nous = pd.DataFrame(sv_rate(T, *[hist_sv[c].to_numpy()[:, None] for c in noms_sv]),
                           index=hist_sv.index, columns=T)
ecart_pb = (courbe_nous - courbe_bce).abs().max(axis=1) * 1e4
print((ecart_pb > 0.05).sum())                       # avant : 77

0
54
0.7767573295655246
39
